# ZelusBench: Sustained Attention

**Does accuracy degrade as dependency chains grow longer?**

Measures sustained attention by varying dependency chain depth.
The math at each step is trivial — the challenge is tracking a growing
chain of point definitions across the prompt.

- Shallow (depth 2-3) | Medium (depth 5-6) | Deep (depth 9-10)
- 5 seeds per depth level = 30 scenarios, 90 queries

In [ ]:
import kaggle_benchmarks as kbench
import numpy as np
import re
import time

from zelusbench.scenarios.config import (
    ScenarioConfig, DAGStructure, QueryType,
    DistractorLevel, TransformDensity,
)
from zelusbench.scenarios.generator import ScenarioGenerator
from zelusbench.evaluation.parser import parse_model_response
from zelusbench.evaluation.scorer import score_query


def score_response(response_text, scenario):
    """Parse model response and score each query against ground truth."""
    query_dicts = [q.to_dict() for q in scenario.queries]
    parts = re.split(r'\[Query\s+q_\d+\]', response_text)
    if len(parts) > 1:
        parts = parts[1:]
    if len(parts) != len(query_dicts):
        parts = [response_text] * len(query_dicts)
    return [score_query(qd, parse_model_response(rp, qd))
            for qd, rp in zip(query_dicts, parts)]


CHAIN_DEPTHS = [2, 3, 5, 6, 9, 10]
SEEDS_PER_DEPTH = 5
TOTAL = len(CHAIN_DEPTHS) * SEEDS_PER_DEPTH

print(f"ZelusBench Sustained Attention")
print(f"Depths: {CHAIN_DEPTHS} | Seeds: {SEEDS_PER_DEPTH} | Total: {TOTAL} scenarios")

In [ ]:
@kbench.task(name="zelusbench_sustained_attention")
def zelusbench_sustained_attention(llm) -> tuple[float, float]:
    """Sustained attention: accuracy vs chain depth.
    Returns (mean_accuracy, std_dev) across all depth levels."""

    _start = time.time()
    print(f"Running {TOTAL} scenarios across {len(CHAIN_DEPTHS)} chain depths...")
    print("=" * 60)

    all_scores = []       # flat list of QueryScore objects
    depth_scores = {}     # depth -> list of per-scenario averages
    scenario_num = 0

    for depth in CHAIN_DEPTHS:
        depth_scores[depth] = []
        for seed in range(SEEDS_PER_DEPTH):
            scenario_num += 1
            # Offset seed by depth*100 so different depths don't share RNG
            unique_seed = depth * 100 + seed
            cfg = ScenarioConfig(
                dim=3,
                min_chain_depth=depth, max_chain_depth=depth,
                dag_structure=DAGStructure.LINEAR,
                distractor_level=DistractorLevel.CLEAN,
                transform_density=TransformDensity.STATIC,
                query_types=[QueryType.POSITION, QueryType.DISTANCE],
                num_queries=3, num_splits=3,
                seed=unique_seed,
            )
            gen = ScenarioGenerator(cfg)
            scenario = gen.generate(f"sustained_d{depth}_s{seed}")

            response = llm.prompt(scenario.prompt)
            scores = score_response(response, scenario)
            all_scores.extend(scores)

            avg = float(np.mean([s.score for s in scores]))
            depth_scores[depth].append(avg)

            q_depths = [s.chain_depth for s in scores]
            tiers = [s.tier.name for s in scores]
            print(f"  [{scenario_num}/{TOTAL}] depth={depth} seed={seed} "
                  f"avg={avg:.2f} q_depths={q_depths} tiers={tiers}")

        depth_avg = float(np.mean(depth_scores[depth]))
        print(f"  >> Depth {depth} mean: {depth_avg:.3f}")

    # Assertions per depth level
    print("\n" + "=" * 60)
    print("RESULTS BY DEPTH:")
    for depth in CHAIN_DEPTHS:
        avg = float(np.mean(depth_scores[depth]))
        print(f"  depth={depth:2d}  accuracy={avg:.3f}")
        kbench.assertions.assert_true(
            avg >= 0,
            expectation=f"Depth {depth}: model should produce valid answers (got {avg:.3f})"
        )

    overall = float(np.mean([s.score for s in all_scores]))
    std = float(np.std([s.score for s in all_scores]))
    elapsed = time.time() - _start

    print(f"\nOverall: {overall:.3f} +/- {std:.3f}")
    print(f"Total queries: {len(all_scores)}")
    print(f"Time: {elapsed:.1f}s ({elapsed/60:.1f} min)")

    return overall, std


zelusbench_sustained_attention.run(llm=kbench.llm)

In [ ]:
%choose zelusbench_sustained_attention